# 🚗 JalanCerdas AI — Training Model v3

YOLOv8s (Small) — 6 kelas deteksi kerusakan jalan

| Kelas | Deskripsi |
|-------|-----------|
| 0 | Retak Memanjang (Longitudinal Crack) |
| 1 | Pengelupasan Lapisan Permukaan (Surface Peeling) |
| 2 | Lubang (Pothole) |
| 3 | Retak Kulit Buaya (Alligator Crack) |
| 4 | Retak Blok (Block Crack) |
| 5 | Retak Pinggir (Edge Crack) |

**⏱️ Estimasi:** ~30-50 menit (GPU T4 Colab)

**📋 Checklist sebelum mulai:**
- [ ] Runtime → Change runtime type → **GPU (T4)**

## Step 1: Setup & Install Dependencies

In [ ]:
#@title Install packages & cek GPU
!pip install -q ultralytics kagglehub pyyaml

import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f'✅ GPU: {gpu} ({mem:.1f} GB)')
else:
    print('❌ TIDAK ADA GPU! Set Runtime → Change runtime type → GPU')
    print('   Training di CPU akan sangat lambat (~10 jam)')

import ultralytics
print(f'📦 Ultralytics: {ultralytics.__version__}')

## Step 2: Clone Repo & Download Dataset

In [ ]:
#@title Clone repo & download dataset dari Kaggle
import os
from pathlib import Path

# Clone repo
if not Path('jalancerdas-ai').exists():
    !git clone -q https://github.com/Srjeff27/jalancerdas-ai.git
    print('✅ Repo cloned')
else:
    print('✅ Repo sudah ada')

# Download dataset
import kagglehub
print('\n📥 Downloading dataset dari Kaggle...')
src = Path(kagglehub.dataset_download('habibiahmadim09/kerusakan-jalan'))
print(f'✅ Downloaded: {src}')

# Fix nested path
if (src / 'kerusakan-jalan').exists():
    src = src / 'kerusakan-jalan'

# Show stats
for split in ['train', 'valid', 'test']:
    d = src / split
    if d.exists():
        n = len(list(d.glob('*.jpg')))
        print(f'  {split}: {n} images')

## Step 3: Konversi VOC → YOLO Format

In [ ]:
#@title Konversi Pascal VOC (XML) ke YOLO (TXT)
import shutil
import xml.etree.ElementTree as ET
import yaml

# Class mapping
CLASS_MAP = {
    'retak_memanjang': 0,
    'pengelupasan_lapisan_permukaan': 1,
    'lubang': 2,
    'retak_kulit_buaya': 3,
    'retak_blok': 4,
    'retak_pinggir': 5,
}

CLASS_NAMES = list(CLASS_MAP.keys())

# Dataset output path
DS = Path('jalancerdas-ai/ai-model/dataset')
if DS.exists():
    shutil.rmtree(DS)

SPLIT_MAP = {'train': 'train', 'valid': 'val', 'test': 'test'}
total_images = 0
total_labels = 0
class_counts = {i: 0 for i in range(6)}

for src_split, dst_split in SPLIT_MAP.items():
    sd = src / src_split
    if not sd.exists():
        print(f'⚠️  {src_split}/ tidak ditemukan')
        continue

    img_dir = DS / dst_split / 'images'
    lbl_dir = DS / dst_split / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    images = list(sd.glob('*.jpg')) + list(sd.glob('*.png'))
    xmls = {x.stem: x for x in sd.glob('*.xml')}

    n_img = 0
    n_lbl = 0

    for img in images:
        shutil.copy2(img, img_dir / img.name)
        n_img += 1

        xml = xmls.get(img.stem)
        if not xml:
            continue

        try:
            tree = ET.parse(xml)
            root = tree.getroot()
            w = int(root.find('size/width').text)
            h = int(root.find('size/height').text)
            lines = []

            for obj in root.findall('object'):
                name = obj.find('name').text
                if name not in CLASS_MAP:
                    continue
                cls = CLASS_MAP[name]
                bbox = obj.find('bndbox')
                x1 = float(bbox.find('xmin').text)
                y1 = float(bbox.find('ymin').text)
                x2 = float(bbox.find('xmax').text)
                y2 = float(bbox.find('ymax').text)

                cx = ((x1 + x2) / 2) / w
                cy = ((y1 + y2) / 2) / h
                bw = (x2 - x1) / w
                bh = (y2 - y1) / h
                lines.append(f'{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
                class_counts[cls] += 1
                n_lbl += 1

            (lbl_dir / (img.stem + '.txt')).write_text('\n'.join(lines))
        except Exception as e:
            pass

    total_images += n_img
    total_labels += n_lbl
    print(f'✅ {dst_split}: {n_img} images, {n_lbl} labels')

# Show class distribution
print(f'\n📊 Distribusi kelas (total: {total_labels} instances):')
max_c = max(class_counts.values())
for cls_id, count in class_counts.items():
    bar = '█' * int(count / max_c * 25) if max_c > 0 else ''
    pct = count / total_labels * 100 if total_labels > 0 else 0
    print(f'  {CLASS_NAMES[cls_id]:<35} {count:>5} ({pct:>5.1f}%) {bar}')

# Update data.yaml with absolute path
yaml_path = Path('jalancerdas-ai/ai-model/configs/data.yaml')
abs_path = str(Path.cwd() / DS)
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = abs_path
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f'\n✅ data.yaml updated: path = {abs_path}')
DATA_YAML = str(yaml_path)

## Step 4: Training 🚀

Model: **YOLOv8s** (Small) — balance antara akurasi dan kecepatan

⏱️ Estimasi: ~30-50 menit di GPU T4

**⚠️ JANGAN TUTUP TAB INI SELAMA TRAINING!**

In [ ]:
#@title Training YOLOv8s — 200 epochs
#@markdown Training akan otomatis berhenti jika tidak ada perbaikan (early stopping)
import time
from ultralytics import YOLO

print('=' * 60)
print('🚗 JalanCerdas AI — Training YOLOv8s')
print('=' * 60)
print(f'📁 Data: {DATA_YAML}')
print(f'🔧 Model: yolov8s.pt')
print(f'📊 Epochs: 200 (early stopping patience: 50)')
print(f'📦 Batch: 16')
print(f'🖼️  Image size: 640')
print(f'💻 Device: GPU (CUDA)')
print('=' * 60)

start = time.time()

model = YOLO('yolov8s.pt')

results = model.train(
    data=DATA_YAML,
    epochs=200,
    batch=16,
    imgsz=640,
    device=0,
    patience=50,          # Early stopping
    
    # Optimizer
    optimizer='AdamW',
    lr0=0.001,            # Lower LR for stability
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.001,
    warmup_epochs=5,
    cos_lr=True,          # Cosine LR schedule
    
    # Augmentation (road damage optimized)
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,       # Handle class imbalance
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    erasing=0.3,
    close_mosaic=15,      # Disable mosaic last 15 epochs
    
    # NMS
    conf=0.25,
    iou=0.6,
    max_det=300,
    
    # Output
    project='jalancerdas-ai/ai-model/runs',
    name='train_v3',
    exist_ok=True,
    verbose=True,
)

elapsed = int(time.time() - start)
BEST_PT = 'jalancerdas-ai/ai-model/runs/train_v3/weights/best.pt'

print(f'\n{"=" * 60}')
print(f'✅ Training selesai dalam {elapsed // 60} menit {elapsed % 60} detik')
print(f'📁 Save dir: jalancerdas-ai/ai-model/runs/train_v3/')
print(f'🏆 Best model: {BEST_PT}')
print(f'{"=" * 60}')

## Step 5: Evaluasi Model

In [ ]:
#@title Evaluasi pada validation set
from ultralytics import YOLO
import json

print('📊 Evaluasi model pada validation set...\n')

model = YOLO(BEST_PT)
results = model.val(data=DATA_YAML, device=0, split='val')

# Metrics
f1 = 2 * (results.box.mp * results.box.mr) / (results.box.mp + results.box.mr + 1e-8)

print(f'{"=" * 50}')
print(f'📈 Hasil Evaluasi:')
print(f'{"=" * 50}')
print(f'  mAP@50:      {results.box.map50:.4f}')
print(f'  mAP@50-95:   {results.box.map:.4f}')
print(f'  Precision:   {results.box.mp:.4f}')
print(f'  Recall:      {results.box.mr:.4f}')
print(f'  F1 Score:    {f1:.4f}')
print(f'{"=" * 50}')

# Target check
print(f'\n🎯 Target Check:')
targets = {'mAP@50': (results.box.map50, 0.70), 'Precision': (results.box.mp, 0.75), 'Recall': (results.box.mr, 0.70)}
for name, (val, target) in targets.items():
    status = '✅' if val >= target else '⚠️'
    print(f'  {status} {name}: {val:.4f} (target: ≥{target})')

# Per-class AP
CLASS_NAMES = ['retak_memanjang', 'pengelupasan_lapisan_permukaan', 'lubang',
               'retak_kulit_buaya', 'retak_blok', 'retak_pinggir']
if hasattr(results.box, 'ap50') and results.box.ap50 is not None:
    print(f'\n📊 Per-class AP@50:')
    for i, ap in enumerate(results.box.ap50):
        if i < len(CLASS_NAMES):
            print(f'  {CLASS_NAMES[i]:<35} {ap:.4f}')

# Save metrics
metrics = {
    'mAP50': float(results.box.map50),
    'mAP50_95': float(results.box.map),
    'precision': float(results.box.mp),
    'recall': float(results.box.mr),
    'f1': float(f1),
}
with open('jalancerdas-ai/ai-model/runs/train_v3/evaluation.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\n💾 Metrics saved: jalancerdas-ai/ai-model/runs/train_v3/evaluation.json')

## Step 6: Export ke TFLite (untuk Mobile)

In [ ]:
#@title Export model ke TFLite FP16
import os
from pathlib import Path

print('📦 Exporting ke TFLite (FP16)...\n')

model = YOLO(BEST_PT)
tflite_path = model.export(
    format='tflite',
    imgsz=640,
    half=True,       # FP16 quantization
    simplify=True,
)

size_mb = os.path.getsize(tflite_path) / (1024 * 1024)

print(f'{"=" * 50}')
print(f'✅ TFLite exported!')
print(f'📁 Path: {tflite_path}')
print(f'📏 Size: {size_mb:.2f} MB')
print(f'{"=" * 50}')

if size_mb > 15:
    print(f'\n⚠️  Model size ({size_mb:.2f} MB) cukup besar untuk mobile')
    print('   Pertimbangkan export INT8 untuk ukuran lebih kecil')

TFLITE_PATH = tflite_path

## Step 7: Download Model 📥

Download `best.pt` dan `best.tflite` ke komputer Anda.

In [ ]:
#@title Download model files
from google.colab import files
from pathlib import Path

print('📥 Downloading files...\n')

# Download best.pt (PyTorch weights)
best_pt = Path(BEST_PT)
if best_pt.exists():
    print(f'  📄 {best_pt.name} ({best_pt.stat().st_size / 1024 / 1024:.2f} MB)')
    files.download(str(best_pt))

# Download best.tflite (mobile model)
tflite = Path(TFLITE_PATH)
if tflite.exists():
    print(f'  📄 {tflite.name} ({tflite.stat().st_size / 1024 / 1024:.2f} MB)')
    files.download(str(tflite))

print(f'\n✅ Done! Cara pakai:')
print(f'   1. Copy best.tflite → mobile/assets/models/pothole_yolo.tflite')
print(f'   2. Build APK: cd mobile && flutter build apk --release')

## 📋 Ringkasan

| Item | Value |
|------|-------|
| Model | YOLOv8s (Small) |
| Epochs | 200 (early stopping) |
| Dataset | 4,004 train / 500 val / 501 test |
| Classes | 6 kelas kerusakan jalan |
| Input | 640×640 RGB |
| Output | TFLite FP16 (~11 MB) |

### 🚀 Next Steps

1. Copy `best.tflite` ke `mobile/assets/models/pothole_yolo.tflite`
2. Build APK: `cd mobile && flutter build apk --release`
3. Test di HP dengan dashcam footage
4. Jika hasil bagus, push ke repo!